# 03_preprocessing.ipynb

Leakage-Safe Preprocessing and Feature Engineering Interactive Notebook.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

from src.utils.config import load_config
from src.utils.reproducibility import set_seed
from src.data.loaders import CICIDS2017Loader, DatasetNotFoundError
from src.data.dataset_adapters import CICIDS2017Adapter
from src.preprocessing.pipeline import CyberthreatPreprocessingPipeline

set_seed(42)
config = load_config("../configs/config.yaml")
print("Initialized Preprocessing & Feature Engineering Environment.")


In [ ]:
# 1. Dataset Loading & Adapter Extraction
try:
    loader = CICIDS2017Loader("../data/raw/CICIDS2017")
    df_raw = loader.load_merged_dataset(sample_frac=0.1)
except DatasetNotFoundError:
    print("[NOTE] Raw dataset absent. Using synthetic dataset for pipeline demonstration...")
    df_raw = pd.DataFrame({
        " Destination Port": np.random.choice([80, 443, 22, 8080], 500),
        " Flow Duration": np.random.exponential(1000, 500),
        " Total Fwd Packets": np.random.randint(1, 50, 500),
        " Total Backward Packets": np.random.randint(0, 50, 500),
        " Total Length of Fwd Packets": np.random.uniform(10, 5000, 500),
        " Total Length of Bwd Packets": np.random.uniform(0, 5000, 500),
        " Flow Bytes/s": np.random.uniform(0, 1e6, 500),
        " Constant Col": [1.0] * 500,
        " Label": np.random.choice(["BENIGN", "DDoS", "PortScan"], 500, p=[0.7, 0.2, 0.1])
    })

adapter = CICIDS2017Adapter()
X, y_bin, y_multi = adapter.extract_labels(df_raw)
print(f"Raw Input Matrix Shape: X={X.shape}, y_binary={y_bin.shape}")


In [ ]:
# 2. Fit Leakage-Safe Preprocessing Pipeline
pipeline = CyberthreatPreprocessingPipeline(
    impute_strategy=config.data.handle_missing,
    scaling_method=config.data.scaling_method,
    encoding_type=config.data.categorical_encoding,
    test_size=config.data.test_size,
    val_size=config.data.val_size,
    random_state=config.system.seed
)

processed = pipeline.fit_transform_splits(X, y_bin, y_multi)
print("Pipeline Transformation Results:")
print(f"- X_train shape: {processed['X_train'].shape}")
print(f"- X_val shape:   {processed['X_val'].shape}")
print(f"- X_test shape:  {processed['X_test'].shape}")
print(f"- Output features ({len(pipeline.feature_names_out_)}): {pipeline.feature_names_out_}")


In [ ]:
# 3. Save Fitted Artifact
artifact_path = Path("../models/artifacts/preprocessing_pipeline.joblib")
pipeline.save_pipeline(artifact_path)
print(f"Artifact saved successfully to: {artifact_path.resolve()}")
